# ERD-Align: ERD-Selective Riemannian Alignment

**Addon notebook** — Chạy SAU khi đã chạy xong `EK_TTA_Experiment.ipynb` (cells 1–9 phải đã được execute để có các function cần thiết).

---
### Framework ERD-Align

Thay vì update model weights (rủi ro), **transform input signal** về không gian phân phối của training subjects:

```
ERD Screening
     ↓
ERD subject?  →  Riemannian Alignment (mu/beta band)  →  Frozen EEGNet → Predict
ERS subject?  →  No-adapt                             →  Frozen EEGNet → Predict
```

**Novelty so với standard Riemannian alignment:**
- Alignment được tính từ mu+beta band covariance (8–30 Hz) — đúng nơi ERD xảy ra
- ERD screening tự động chọn subjects phù hợp để align
- Không cần label, không cần gradient, edge-compatible

In [ ]:
# ── Cell 15: Signal Processing Utilities ─────────────────
# Chạy cell này trước. Cần scipy (đã có trong Colab).

from scipy.signal import butter, filtfilt
from scipy.linalg import sqrtm, inv
import numpy as np
import warnings
warnings.filterwarnings('ignore')


def bandpass_filter_batch(X, low, high, sfreq=250, order=4):
    """
    Bandpass filter cho batch EEG.
    X: (n_trials, n_channels, n_times)
    Returns: X_filtered cùng shape
    """
    nyq = sfreq / 2
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    X_filt = np.zeros_like(X)
    for i in range(len(X)):
        for j in range(X.shape[1]):
            X_filt[i, j] = filtfilt(b, a, X[i, j])
    return X_filt


def compute_mean_cov(X_filt, reg=1e-5):
    """
    Tính mean covariance matrix từ batch trials.
    X_filt: (n_trials, n_channels, n_times)
    Returns: (n_channels, n_channels) — regularized SPD matrix
    """
    covs = []
    for x in X_filt:
        n   = x.shape[-1]
        cov = (x @ x.T) / n
        cov += np.eye(cov.shape[0]) * reg  # regularization
        covs.append(cov)
    return np.mean(covs, axis=0)


def safe_sqrt(M):
    """Matrix square root, lấy phần thực (sqrtm trả về complex)."""
    return np.real(sqrtm(M))


def safe_sqrt_inv(M):
    """Matrix inverse square root."""
    S = safe_sqrt(M)
    return np.real(inv(S))


def compute_whitening_transform(sigma_source, sigma_target):
    """
    Tính whitening + recoloring transform:
        W = sigma_target^(1/2) @ sigma_source^(-1/2)

    Applied as X_aligned = W @ X_source
    → X_aligned có covariance gần với sigma_target

    Ref: He & Wu (2019) "Transfer Learning for BCI" — Euclidean Alignment
    ERD-selective version: sigma được tính từ mu/beta band filtered signal
    """
    sqrt_target     = safe_sqrt(sigma_target)
    sqrt_inv_source = safe_sqrt_inv(sigma_source)
    W = sqrt_target @ sqrt_inv_source
    return np.real(W)


def riemannian_mean_iterative(covs, n_iter=15, tol=1e-8):
    """
    Riemannian mean của SPD matrices qua gradient descent trên SPD manifold.
    Dùng cho source reference (offline, chạy 1 lần).

    M_{k+1} = M_k^(1/2) exp(1/n Σ log(M_k^(-1/2) C_i M_k^(-1/2))) M_k^(1/2)
    """
    from scipy.linalg import logm, expm

    n = len(covs)
    M = np.mean(covs, axis=0)          # Euclidean init

    for it in range(n_iter):
        M_sqrt     = safe_sqrt(M)
        M_sqrt_inv = safe_sqrt_inv(M)

        grad = np.zeros_like(M)
        for C in covs:
            inner  = M_sqrt_inv @ C @ M_sqrt_inv
            grad  += np.real(logm(inner))
        grad /= n

        M_new = M_sqrt @ np.real(expm(grad)) @ M_sqrt
        diff  = np.linalg.norm(M_new - M, 'fro')
        M     = M_new
        if diff < tol:
            break

    return M


print('✓ Signal processing utilities loaded')
print('  bandpass_filter_batch | compute_mean_cov | compute_whitening_transform')
print('  riemannian_mean_iterative')

In [ ]:
# ── Cell 16: ERD-Align Method ─────────────────────────────
import torch
import numpy as np

class ERDAlignMethod:
    """
    ERD-Selective Riemannian Alignment cho MI EEG.

    Pipeline:
    1. fit_source(): tính reference covariance từ source subjects (offline)
    2. screen_and_fit_test(): đánh giá ERD/ERS + tính alignment transform
    3. predict_all(): predict với frozen model (+ alignment nếu ERD)

    Key difference với EK-TTA:
    - KHÔNG update model weights
    - Transform INPUT signal về distribution của source subjects
    - Alignment driven bởi mu/beta band (8-30Hz) — nơi ERD xảy ra
    """

    def __init__(self, model, cfg, c3_idx, c4_idx,
                 band_low=8, band_high=30,
                 erd_threshold=-10.0,
                 use_riemannian=True):
        """
        Args:
            model         : EEGNet đã train (sẽ bị frozen)
            cfg           : CONFIG dict
            c3_idx/c4_idx : channel indices
            band_low/high : ERD frequency band (default: 8-30Hz)
            erd_threshold : ngưỡng phân loại ERD vs ERS
            use_riemannian: True = Riemannian mean (chính xác hơn, chậm hơn)
                            False = Euclidean mean (nhanh hơn, vẫn ổn)
        """
        self.model         = model
        self.cfg           = cfg
        self.c3_idx        = c3_idx
        self.c4_idx        = c4_idx
        self.band_low      = band_low
        self.band_high     = band_high
        self.erd_threshold = erd_threshold
        self.use_riemannian= use_riemannian

        # Computed at fit time
        self.sigma_ref   = None   # source reference covariance
        self.W           = None   # alignment transform (None = no align)
        self.has_erd     = None
        self.erd_value   = None

    # ─── Source fitting (offline, once per LOSO fold) ────────

    def fit_source(self, X_source_model, sfreq=250):
        """
        Tính reference covariance từ source subjects.

        X_source_model: (n_trials, n_ch, n_times_task) — task period [0,4s]

        Strategy: hierarchical averaging
          Per-trial covariances → mean covariance
          Sử dụng Euclidean mean hoặc Riemannian mean
        """
        # Filter to ERD band
        X_filt = bandpass_filter_batch(
            X_source_model, self.band_low, self.band_high, sfreq)

        # Compute per-trial covariances
        n_ch = X_filt.shape[1]
        covs = []
        for x in X_filt:
            c = (x @ x.T) / x.shape[-1]
            c += np.eye(n_ch) * 1e-5
            covs.append(c)

        if self.use_riemannian:
            # Riemannian mean — chính xác, dùng cho paper cuối
            # Subsample nếu quá nhiều trials (speed)
            max_cov = 200
            if len(covs) > max_cov:
                idx  = np.random.choice(len(covs), max_cov, replace=False)
                covs = [covs[i] for i in idx]
            self.sigma_ref = riemannian_mean_iterative(covs)
        else:
            # Euclidean mean — nhanh, cho debugging
            self.sigma_ref = np.mean(covs, axis=0)

    # ─── Test fitting (per subject) ──────────────────────────

    def screen_and_fit_test(self, X_test_full, X_test_model, sfreq=250):
        """
        Bước 1: ERD Screening — phân loại subject là ERD hay ERS.
        Bước 2: Nếu ERD, tính alignment transform W.

        X_test_full  : (n_trials, n_ch, 1500) — full epoch với baseline
        X_test_model : (n_trials, n_ch, 1000) — task period
        """
        assert self.sigma_ref is not None, \
            "Chạy fit_source() trước!"

        # ── ERD Screening ──────────────────────────────────
        erds = [compute_erd(
                    X_test_full[i], self.c3_idx, self.c4_idx, self.cfg)
                for i in range(len(X_test_full))]

        # Dùng composite ERD (trung bình C3 và C4)
        # NOTE: Vì không có nhãn, dùng average cả hai kênh
        # ERD âm ở BẤT KỲ kênh nào → khả năng có ERD
        c3_vals = [e['C3_comp'] for e in erds]
        c4_vals = [e['C4_comp'] for e in erds]
        self.erd_value = (np.mean(c3_vals) + np.mean(c4_vals)) / 2

        # Criterion: min(C3, C4) < threshold (ít nhất 1 kênh có ERD)
        min_erd = min(np.mean(c3_vals), np.mean(c4_vals))
        self.has_erd = min_erd < self.erd_threshold

        if not self.has_erd:
            self.W = None  # no alignment → no-adapt
            return

        # ── Compute alignment transform ─────────────────────
        # Filter test data to ERD band
        X_filt = bandpass_filter_batch(
            X_test_model, self.band_low, self.band_high, sfreq)

        # Test covariance (mean over all test trials)
        sigma_test = compute_mean_cov(X_filt)

        # Whitening: map test distribution → source distribution
        # W = sigma_ref^(1/2) @ sigma_test^(-1/2)
        self.W = compute_whitening_transform(sigma_test, self.sigma_ref)

    # ─── Prediction ──────────────────────────────────────────

    def predict(self, x_raw_np, x_model_np, device):
        """
        Predict cho 1 trial.

        x_model_np: (1, n_ch, n_times) numpy array
        """
        if self.W is not None:
            # Apply alignment: W @ x  (channel × time)
            x_aligned = self.W @ x_model_np[0]   # (n_ch, n_times)
            x_t = torch.FloatTensor(
                x_aligned[np.newaxis, np.newaxis]).to(device)
        else:
            x_t = torch.FloatTensor(
                x_model_np[:, np.newaxis]).to(device)

        self.model.eval()
        with torch.no_grad():
            return self.model(x_t).argmax(-1).item(), self.erd_value


print('✓ ERDAlignMethod defined')
print('  fit_source() | screen_and_fit_test() | predict()')

In [ ]:
# ── Cell 17: Updated LOSO — tất cả 5 methods ─────────────
# Thêm ERD-Align và ERD-Align+EK vào LOSO loop

import copy
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score


def run_loso_full(subject_data, cfg, device, ds_name,
                  use_riemannian=True):
    """
    LOSO với 5 methods:
      1. no_adapt
      2. tent
      3. ek_tta
      4. erd_align        ← NEW
      5. erd_align_ek     ← NEW (ERD-Align + EK fine-tuning)

    ERD-Align là method chính cần benchmark.
    ERD-Align+EK kết hợp hai approach (optional ablation).
    """
    subjects = sorted(subject_data.keys())
    rows     = []

    for i, test_subj in enumerate(subjects):
        print(f"\n  Fold {i+1}/{len(subjects)} — Test: {test_subj}")

        info   = subject_data[test_subj]
        c3, c4 = info['c3_idx'], info['c4_idx']
        li, ri = info['left_idx'], info['right_idx']

        # Source data
        src_X = np.concatenate(
            [subject_data[s]['X_model'] for s in subjects if s != test_subj])
        src_y = np.concatenate(
            [subject_data[s]['y']       for s in subjects if s != test_subj])
        n_ch, n_t = src_X.shape[1], src_X.shape[2]

        # Train backbone
        print(f"    Training EEGNet...", end=' ')
        model      = train_backbone(src_X, src_y, n_ch, n_t, cfg, device)
        orig_state = copy.deepcopy(model.state_dict())
        print('done')

        X_full  = info['X_full']    # (n_trials, n_ch, 1500)
        X_model = info['X_model']   # (n_trials, n_ch, 1000)
        y_true  = info['y']

        # ── 1. No-adapt ──────────────────────────────────────
        model.load_state_dict(orig_state); model.eval()
        with torch.no_grad():
            xt = torch.FloatTensor(X_model[:, np.newaxis]).to(device)
            preds_na = model(xt).argmax(-1).cpu().numpy()

        # ── 2. TENT ──────────────────────────────────────────
        model.load_state_dict(orig_state)
        tent_m = TENTMethod(model, cfg)
        preds_tent = []
        for j in range(len(X_model)):
            xt = torch.FloatTensor(X_model[j:j+1, np.newaxis]).to(device)
            p, _ = tent_m.predict(None, xt)
            preds_tent.append(p)
        preds_tent = np.array(preds_tent)

        # ── 3. EK-TTA ────────────────────────────────────────
        model.load_state_dict(orig_state)
        ek_m = EKTTAMethod(model, cfg, c3, c4, li, ri)
        preds_ek, erds_ek = [], []
        for j in range(len(X_model)):
            xt = torch.FloatTensor(X_model[j:j+1, np.newaxis]).to(device)
            p, erd = ek_m.predict(X_full[j:j+1], xt)
            preds_ek.append(p); erds_ek.append(erd)
        preds_ek = np.array(preds_ek)

        # ── 4. ERD-Align ─────────────────────────────────────
        model.load_state_dict(orig_state)
        align_m = ERDAlignMethod(
            model, cfg, c3, c4,
            band_low=cfg['mu_low'], band_high=cfg['beta_high'],
            erd_threshold=cfg['erd_threshold'],
            use_riemannian=use_riemannian)

        print(f"    ERD-Align fitting source...", end=' ')
        align_m.fit_source(src_X, sfreq=cfg['sfreq'])

        align_m.screen_and_fit_test(X_full, X_model, sfreq=cfg['sfreq'])
        erd_group = "ERD" if align_m.has_erd else "ERS"
        print(f"    Subject group: {erd_group} (ERD={align_m.erd_value:.1f}%)")

        preds_align, erds_align = [], []
        for j in range(len(X_model)):
            p, ev = align_m.predict(X_full[j:j+1], X_model[j:j+1], device)
            preds_align.append(p); erds_align.append(ev)
        preds_align = np.array(preds_align)

        # ── 5. ERD-Align + EK fine-tuning ────────────────────
        # Sau khi align, áp dụng EK-TTA nhẹ trên aligned signal
        model.load_state_dict(orig_state)
        align_ek_m = ERDAlignMethod(
            model, cfg, c3, c4,
            band_low=cfg['mu_low'], band_high=cfg['beta_high'],
            erd_threshold=cfg['erd_threshold'],
            use_riemannian=use_riemannian)
        align_ek_m.fit_source(src_X, sfreq=cfg['sfreq'])
        align_ek_m.screen_and_fit_test(X_full, X_model, sfreq=cfg['sfreq'])

        # EK-TTA trên top của aligned signal
        ek_on_align = EKTTAMethod(model, cfg, c3, c4, li, ri)
        preds_align_ek = []
        for j in range(len(X_model)):
            # Step 1: align
            if align_ek_m.W is not None:
                x_a = align_ek_m.W @ X_model[j]   # (n_ch, n_times)
                x_model_j = x_a[np.newaxis]
            else:
                x_model_j = X_model[j:j+1]
            # Step 2: EK-TTA update
            xt = torch.FloatTensor(x_model_j[:, np.newaxis]).to(device)
            p, _ = ek_on_align.predict(X_full[j:j+1], xt)
            preds_align_ek.append(p)
        preds_align_ek = np.array(preds_align_ek)

        # ── ERD mean per subject ──────────────────────────────
        erd_means = {f'erd_{k}': np.mean([e[k] for e in erds_ek])
                     for k in erds_ek[0].keys()}

        # ── Log ──────────────────────────────────────────────
        for method, preds in [
            ('no_adapt',      preds_na),
            ('tent',          preds_tent),
            ('ek_tta',        preds_ek),
            ('erd_align',     preds_align),
            ('erd_align_ek',  preds_align_ek),
        ]:
            acc   = accuracy_score(y_true, preds)
            kappa = cohen_kappa_score(y_true, preds)
            row   = {'dataset': ds_name, 'subject': test_subj,
                     'method': method,
                     'accuracy': acc, 'kappa': kappa,
                     'erd_group': erd_group,
                     'erd_value': align_m.erd_value}
            row.update(erd_means)
            rows.append(row)
            print(f"    {method:<16} acc={acc:.4f}  kappa={kappa:.4f}")

    return pd.DataFrame(rows)


print('✓ run_loso_full() defined (5 methods including ERD-Align)')

In [ ]:
# ── Cell 18: RUN — ERD-Align experiment ──────────────────
# Cần Google Drive đã mount và subject_data đã load.
# Nếu chưa có subject_data, chạy lại Cell 10 của notebook gốc trước.

import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except:
    pass

os.makedirs(CONFIG['output_dir'], exist_ok=True)

all_dfs_v2 = []

for ds_name in ['BNCI2014_004', 'BNCI2014_001']:
    print(f"\n{'='*60}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*60}")

    # Load data
    subj_data, _ = load_mi_dataset(ds_name, CONFIG)

    # Chạy LOSO với 5 methods
    # use_riemannian=False để nhanh hơn khi debug
    # use_riemannian=True  cho kết quả cuối (paper)
    df = run_loso_full(
        subj_data, CONFIG, device, ds_name,
        use_riemannian=False   # ← đổi thành True cho paper
    )

    path = f"{CONFIG['output_dir']}/{ds_name}_v2_results.csv"
    df.to_csv(path, index=False)
    print(f"  ✓ Saved → {path}")
    all_dfs_v2.append(df)

df_v2 = pd.concat(all_dfs_v2, ignore_index=True)
df_v2.to_csv(f"{CONFIG['output_dir']}/all_results_v2.csv", index=False)
print(f"\n✓ All v2 results saved")

In [ ]:
# ── Cell 19: Summary Table (5 methods) ───────────────────
from scipy.stats import wilcoxon

methods_v2 = ['no_adapt', 'tent', 'ek_tta', 'erd_align', 'erd_align_ek']
labels_v2  = ['No-adapt', 'TENT', 'EK-TTA', 'ERD-Align (ours)', 'ERD-Align+EK (ours)']

print("TABLE 2 — Full comparison (mean ± std, n=9 subjects)")
print("="*68)

for ds in ['BNCI2014_004', 'BNCI2014_001']:
    df = df_v2[df_v2['dataset'] == ds]
    print(f"\n  {ds}")
    print(f"  {'Method':<22} {'Acc':>8} {'± std':>7} {'Kappa':>8} {'vs NA':>8}")
    print(f"  {'-'*58}")

    na_mean = df[df['method']=='no_adapt']['accuracy'].mean()

    for m, label in zip(methods_v2, labels_v2):
        sub  = df[df['method'] == m]['accuracy']
        diff = sub.mean() - na_mean
        marker = ' ↑' if diff > 0.002 else (' ↓' if diff < -0.002 else '  ')
        print(f"  {label:<22} {sub.mean():>8.4f} {sub.std():>7.4f}"
              f" {df[df['method']==m]['kappa'].mean():>8.4f}"
              f" {diff:>+7.4f}{marker}")

print("\n" + "-"*68)
print("WILCOXON TESTS — ERD-Align vs No-adapt / vs TENT")
print("-"*68)

for ds in ['BNCI2014_004', 'BNCI2014_001']:
    df   = df_v2[df_v2['dataset'] == ds]
    subs = sorted(df['subject'].unique())

    def get_acc(method):
        return [df[(df['subject']==s)&(df['method']==method)]['accuracy'].values[0]
                for s in subs]

    align_a = get_acc('erd_align')
    na_a    = get_acc('no_adapt')
    tent_a  = get_acc('tent')

    delta_na   = np.array(align_a) - np.array(na_a)
    delta_tent = np.array(align_a) - np.array(tent_a)

    print(f"\n  {ds}:")
    for label, delta, pair in [
        ('ERD-Align vs No-adapt', delta_na,   (align_a, na_a)),
        ('ERD-Align vs TENT',     delta_tent, (align_a, tent_a))
    ]:
        if np.all(delta == 0):
            print(f"    {label}: identical")
            continue
        try:
            stat, p = wilcoxon(pair[0], pair[1])
            d   = delta.mean() / (delta.std() + 1e-10)
            sig = '✓ p<0.05' if p < 0.05 else '✗ ns'
            print(f"    {label}: W={stat:.0f}, p={p:.4f}, "
                  f"d={d:.2f}, Δ={delta.mean():+.4f}  {sig}")
        except Exception as e:
            print(f"    {label}: {e}")

In [ ]:
# ── Cell 20: ERD-Group breakdown ─────────────────────────
# Phân tích theo nhóm ERD vs ERS

print("GROUP ANALYSIS — ERD vs ERS subjects")
print("="*65)

for ds in ['BNCI2014_004', 'BNCI2014_001']:
    df = df_v2[df_v2['dataset'] == ds]
    print(f"\n  {ds}")

    for group in ['ERD', 'ERS']:
        df_g = df[df['erd_group'] == group]
        if len(df_g) == 0: continue
        n_subj = df_g['subject'].nunique()
        print(f"\n  Group {group} ({n_subj} subjects):")
        print(f"  {'Method':<22} {'Acc mean':>10} {'vs NA':>8}")
        print(f"  {'-'*44}")

        na_g = df_g[df_g['method']=='no_adapt']['accuracy'].mean()
        for m, label in zip(methods_v2, labels_v2):
            sub  = df_g[df_g['method']==m]['accuracy']
            if len(sub) == 0: continue
            diff = sub.mean() - na_g
            marker = ' ↑' if diff > 0.005 else (' ↓' if diff < -0.005 else '  ')
            print(f"  {label:<22} {sub.mean():>10.4f} {diff:>+8.4f}{marker}")

# Key question: ERD-Align có beat no-adapt trên ERD subjects không?
print("\n" + "─"*65)
print("KEY QUESTION: ERD-Align > No-adapt trên ERD subjects?")
for ds in ['BNCI2014_004', 'BNCI2014_001']:
    df   = df_v2[df_v2['dataset'] == ds]
    df_erd = df[df['erd_group'] == 'ERD']
    if len(df_erd) == 0: continue
    subs  = sorted(df_erd['subject'].unique())
    align = [df_erd[(df_erd['subject']==s)&(df_erd['method']=='erd_align')]['accuracy'].values[0]
             for s in subs]
    na    = [df_erd[(df_erd['subject']==s)&(df_erd['method']=='no_adapt')]['accuracy'].values[0]
             for s in subs]
    win   = sum(1 for a, n in zip(align, na) if a > n)
    delta = np.array(align) - np.array(na)
    print(f"  {ds}: {win}/{len(subs)} ERD subjects have Align>NA "
          f"(Δmean={delta.mean():+.4f})")

In [ ]:
# ── Cell 21: Visualization ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

palette = {
    'no_adapt'    : '#888780',
    'tent'        : '#E24B4A',
    'ek_tta'      : '#0F6E56',
    'erd_align'   : '#534AB7',
    'erd_align_ek': '#BA7517',
}

for row, ds in enumerate(['BNCI2014_004', 'BNCI2014_001']):
    df  = df_v2[df_v2['dataset'] == ds]
    subs = sorted(df['subject'].unique())

    # ── Bar chart (mean ± std) ────────────────────────
    ax = axes[row, 0]
    ms = [df[df['method']==m]['accuracy'].mean() for m in methods_v2]
    ss = [df[df['method']==m]['accuracy'].std()  for m in methods_v2]
    cs = [palette[m] for m in methods_v2]

    bars = ax.bar(labels_v2, ms, yerr=ss, color=cs, alpha=0.87,
                  capsize=5, width=0.6,
                  error_kw={'lw': 1.5})
    ax.set_ylim(max(0, min(ms)-0.12), min(1.0, max(ms)+0.1))
    ax.set_title(f'{ds} — Mean accuracy', fontsize=11)
    ax.set_ylabel('Accuracy'); ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(labels_v2, rotation=20, ha='right', fontsize=9)
    for bar, m in zip(bars, ms):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{m:.3f}', ha='center', fontsize=8)

    # ── Per-subject: ERD-Align vs No-adapt ───────────
    ax = axes[row, 1]
    align_acc = [df[(df['subject']==s)&(df['method']=='erd_align')]['accuracy'].values[0]
                 for s in subs]
    na_acc    = [df[(df['subject']==s)&(df['method']=='no_adapt')]['accuracy'].values[0]
                 for s in subs]
    erd_groups = [df[(df['subject']==s)]['erd_group'].values[0] for s in subs]

    gc = {'ERD': '#534AB7', 'ERS': '#E24B4A'}
    for s, al, na, g in zip(subs, align_acc, na_acc, erd_groups):
        ax.scatter(na, al, c=gc.get(g, '#888780'), s=80, zorder=5)
        ax.annotate(f'S{s}', (na, al), xytext=(4, 3),
                    textcoords='offset points', fontsize=8)

    lim = [min(na_acc+align_acc)-0.02, max(na_acc+align_acc)+0.02]
    ax.plot(lim, lim, 'k--', lw=0.8, alpha=0.4, label='y=x (equal)')
    ax.set_xlabel('No-adapt accuracy', fontsize=10)
    ax.set_ylabel('ERD-Align accuracy', fontsize=10)
    ax.set_title(f'{ds} — ERD-Align vs No-adapt\n(above diagonal = improvement)',
                 fontsize=10)
    ax.legend(handles=[
        mpatches.Patch(color='#534AB7', label='ERD subjects'),
        mpatches.Patch(color='#E24B4A', label='ERS subjects'),
    ], fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle('ERD-Align: complete results', fontsize=13, y=1.01)
plt.tight_layout()
out = f"{CONFIG['output_dir']}/ERD_Align_results.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out}")

## Đọc kết quả Cell 19–21

### Kịch bản tốt (SOTA):
```
ERD-Align > No-adapt trên ERD subjects    → contribution chính
ERD-Align >> TENT trên tất cả            → đã confirmed
Scatter plot: nhiều điểm TRÊN diagonal   → alignment giúp ích
```

### Kịch bản trung bình:
```
ERD-Align ≈ No-adapt trên ERD subjects   → alignment không đủ mạnh
→ Thử: tăng regularization, dùng Riemannian mean thật (use_riemannian=True)
→ Thử: chỉ align C3/C4 thay vì tất cả kênh
```

### Nếu kết quả không như kỳ vọng:
```
Chạy lại Cell 18 với use_riemannian=True
```

---
**Files output:**
```
EK_TTA_results/
├── BNCI2014_004_v2_results.csv
├── BNCI2014_001_v2_results.csv
├── all_results_v2.csv
└── ERD_Align_results.png
```